In [1]:
import numpy as np
import pandas as pd
import pickle as pc
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB, ComplementNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score

In [2]:
Dataset = pd.read_csv("CKD.csv")
Dataset = pd.get_dummies(Dataset, dtype = int, drop_first = True )
Dataset['classification_yes'].value_counts()

classification_yes
1    249
0    150
Name: count, dtype: int64

In [3]:
Dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='str')

In [4]:
Independent = Dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
Dependent = Dataset['classification_yes']
print("Independent shape :" ,Independent.shape)
print("Dependent shape :" ,Dependent.shape)

Independent shape : (399, 27)
Dependent shape : (399,)


In [5]:
X_Train, X_Test, Y_Train, Y_Test = train_test_split(Independent, Dependent, test_size = 0.30, random_state = 0)

In [6]:
# SCALERS
#
scaler_standard = StandardScaler()
X_Train = scaler_standard.fit_transform(X_Train)
X_Test = scaler_standard.transform(X_Test)

# Multinomial, Complement, Bernoulli
# Need positive values

scaler_minmax = MinMaxScaler()
X_Train_M = scaler_minmax.fit_transform(X_Train)
X_Test_M = scaler_minmax.transform(X_Test)


In [7]:
# MODEL LIST
models = {
    
    "LogisticRegression": (
        LogisticRegression(),
        {
            'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
            'penalty' : ['l2']
        },
        X_Train,
        X_Test,
        scaler_standard
    ),
    
    "SVC": (
        SVC(probability=True),
        {
            'kernel' : ['poly','rbf','sigmoid','linear'] ,
            'C' : [0.1,1,100,500,1000,2000,3000] ,
            'gamma' : ['auto','scale']
        },
        
        X_Train,
        X_Test,
        scaler_standard
    ),

    "DecisionTreeClassifier": (
        DecisionTreeClassifier(),
        {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'splitter': ['best', 'random'],
            'max_features': ['sqrt', 'log2', None]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),

     "RandomForestClassifier": (
        RandomForestClassifier(),
         {
             'n_estimators': [50, 100, 200, 300],
             'criterion': ['gini', 'entropy', 'log_loss'],
             'max_depth': [None, 5, 10, 20, 30],
             'max_features': ['sqrt', 'log2', None]
        },
         X_Train,
         X_Test,
         scaler_standard
    ),   

     "KNeighborsClassifier": (
        KNeighborsClassifier(),
         {
             'n_neighbors': [3, 5, 7, 9, 11],
             'metric': ['euclidean', 'manhattan', 'minkowski'],
             'p': [1, 2]
        },
         X_Train,
         X_Test,
         scaler_standard
    ),   
    
    "GaussianNB": (
        GaussianNB(),
        {
            'var_smoothing': [
                1e-09,
                1e-08,
                1e-07,
                1e-06,
                1e-05
            ]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),

    "MultinomialNB": (
        MultinomialNB(),
        {
            'alpha': [0.01, 0.1, 1, 10, 100],
            'fit_prior': [True, False]
        },
        X_Train_M,
        X_Test_M,
        scaler_minmax
    ),

    "BernoulliNB": (
        BernoulliNB(),
        {
            'alpha': [0.01, 0.1, 1, 10],
            'fit_prior': [True, False]
        },
        X_Train_M,
        X_Test_M,
        scaler_minmax
    ),

    "ComplementNB": (
        ComplementNB(),
        {
            'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
            'fit_prior': [True, False],
            'norm': [True, False]
        },
        X_Train_M,
        X_Test_M,
        scaler_minmax
    ),

    "AdaBoostClassifier": (
        AdaBoostClassifier(),
        {
            'n_estimators': [50, 100, 200, 300],
            'learning_rate': [0.01, 0.1, 0.5, 1.0]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),

    "GradientBoostingClassifier": (
        GradientBoostingClassifier(),
        {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'subsample': [0.8, 1.0]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),


    "XGBClassifier": (
        XGBClassifier(),
        {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0],
            'gamma': [0, 0.1, 0.2],
            'min_child_weight': [1, 3, 5]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),    
 
    "LGBMClassifier": (
        LGBMClassifier(verbose=-1),
        {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [-1, 5, 10],
            'num_leaves': [31, 50, 100],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        },
        X_Train,
        X_Test,
        scaler_standard
    ),    
 
    "CatBoostClassifier": (
        CatBoostClassifier(verbose=0),
        {
            'iterations': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'depth': [4, 6, 8, 10],
            'l2_leaf_reg': [1, 3, 5, 7]
        },
        X_Train,
        X_Test,
        scaler_standard
    )
    
}


In [8]:
# MODEL COMPARISON
import warnings
warnings.filterwarnings('ignore')

Results = []

best_model = None
best_scaler = None
best_f1 = 0

for model_name, (model, params, Xtrain, Xtest,scaler_used) in models.items():

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    Grid_Classifier = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring='f1_weighted',
        cv=5,
        n_jobs=-1,
        refit=True
    )

    Grid_Classifier.fit(Xtrain, Y_Train)

    grid_Y_Pred = Grid_Classifier.predict(Xtest)

    fl_macro_avg = f1_score(
        Y_Test,
        grid_Y_Pred,
        average='weighted'
    )

    auc = roc_auc_score(
        Y_Test,
        Grid_Classifier.predict_proba(Xtest)[:, 1]
    )

    print("Best Parameters        :", Grid_Classifier.best_params_)
    print("F1 Score               :", fl_macro_avg)
    print("ROC AUC                :", auc)
    print("Confusion Matrix       :\n", confusion_matrix(Y_Test, grid_Y_Pred))
    print("Classification Report  :\n", classification_report(Y_Test, grid_Y_Pred))
 

    Results.append(
        [model_name,
         fl_macro_avg,
         auc,
         Grid_Classifier.best_params_]
    )

    if fl_macro_avg > best_f1:

        best_f1 = fl_macro_avg
        best_model = Grid_Classifier.best_estimator_
        best_scaler = scaler_used


LogisticRegression
Best Parameters        : {'penalty': 'l2', 'solver': 'lbfgs'}
F1 Score               : 0.9916844900066377
ROC AUC                : 1.0
Confusion Matrix       :
 [[45  0]
 [ 1 74]]
Classification Report  :
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        45
           1       1.00      0.99      0.99        75

    accuracy                           0.99       120
   macro avg       0.99      0.99      0.99       120
weighted avg       0.99      0.99      0.99       120


SVC
Best Parameters        : {'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}
F1 Score               : 0.9916844900066377
ROC AUC                : 0.9997037037037036
Confusion Matrix       :
 [[45  0]
 [ 1 74]]
Classification Report  :
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        45
           1       1.00      0.99      0.99        75

    accuracy                           0.99   

In [9]:
# RESULT TABLE
Result_DF = pd.DataFrame(
    Results,
    columns=[
        'Model',
        'F1 Score',
        'ROC_AUC',
        'Best Parameters'
    ]
)

print("\n")
print(Result_DF.sort_values(
    by='F1 Score',
    ascending=False
))

print("\nBest Model Selected")
print(best_model)
print("Best F1 Score :", best_f1)



                         Model  F1 Score   ROC_AUC  \
10  GradientBoostingClassifier  1.000000  1.000000   
9           AdaBoostClassifier  1.000000  1.000000   
13          CatBoostClassifier  0.991684  1.000000   
12              LGBMClassifier  0.991684  1.000000   
0           LogisticRegression  0.991684  1.000000   
1                          SVC  0.991684  0.999704   
6                MultinomialNB  0.983402  1.000000   
8                 ComplementNB  0.983402  1.000000   
11               XGBClassifier  0.983333  0.999111   
3       RandomForestClassifier  0.983333  0.999704   
5                   GaussianNB  0.975148  1.000000   
7                  BernoulliNB  0.975148  0.996741   
2       DecisionTreeClassifier  0.958711  0.966667   
4         KNeighborsClassifier  0.950521  0.999556   

                                      Best Parameters  
10  {'learning_rate': 0.1, 'max_depth': 3, 'min_sa...  
9         {'learning_rate': 0.5, 'n_estimators': 300}  
13  {'depth': 6, 'i

In [10]:
pc.dump(best_model,open("MLC_Assignment_best_Model_Grid.sav",'wb'))
pc.dump(best_scaler,open("MLC_Assignment_best_scaler.sav",'wb'))


In [11]:
Loaded_model = pc.load(open ("MLC_Assignment_best_Model_Grid.sav",'rb'))
Loaded_scaler = pc.load(open ("MLC_Assignment_best_scaler.sav",'rb'))


In [14]:
# Numerical Inputs
age = float(input("Age: "))
bp = float(input("BP: "))
al = float(input("Albumin (al): "))
su = float(input("Sugar (su): "))
bgr = float(input("Blood Glucose Random (bgr): "))
bu = float(input("Blood Urea (bu): "))
sc = float(input("Serum Creatinine (sc): "))
sod = float(input("Sodium (sod): "))
pot = float(input("Potassium (pot): "))
hrmo = float(input("Hemoglobin (hrmo): "))
pcv = float(input("Packed Cell Volume (pcv): "))
wc = float(input("White Blood Cell Count (wc): "))
rc = float(input("Red Blood Cell Count (rc): "))

# One-Hot Encoded Inputs
sg_b = int(input("sg_b (0/1): "))
sg_c = int(input("sg_c (0/1): "))
sg_d = int(input("sg_d (0/1): "))
sg_e = int(input("sg_e (0/1): "))

rbc_normal = int(input("rbc_normal (0/1): "))
pc_normal = int(input("pc_normal (0/1): "))
pcc_present = int(input("pcc_present (0/1): "))
ba_present = int(input("ba_present (0/1): "))

htn_yes = int(input("htn_yes (0/1): "))
dm_yes = int(input("dm_yes (0/1): "))
cad_yes = int(input("cad_yes (0/1): "))

appet_yes = int(input("appet_yes (0/1): "))
pe_yes = int(input("pe_yes (0/1): "))
ane_yes = int(input("ane_yes (0/1): "))

# Create DataFrame
new_data = pd.DataFrame(
    [[
        age, bp, al, su, bgr, bu, sc, sod, pot, hrmo,
        pcv, wc, rc,
        sg_b, sg_c, sg_d, sg_e,
        rbc_normal, pc_normal,
        pcc_present, ba_present,
        htn_yes, dm_yes, cad_yes,
        appet_yes, pe_yes, ane_yes
    ]],
    columns=[
        'age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo',
        'pcv', 'wc', 'rc',
        'sg_b', 'sg_c', 'sg_d', 'sg_e',
        'rbc_normal', 'pc_normal',
        'pcc_present', 'ba_present',
        'htn_yes', 'dm_yes', 'cad_yes',
        'appet_yes', 'pe_yes', 'ane_yes'
    ]
)

# Scale data
Preinput = Loaded_scaler.transform(new_data)

# Predict
Future_Predict=Loaded_model.predict(Preinput) # change the paramter,play with it.
probability = Loaded_model.predict_proba(Preinput)
if Future_Predict[0] == 1:
    print("\nPrediction: YES, Patient is likely to have CKD")
    print(f"Confidence : {probability[0][1]*100:.2f}%")
else:
    print("\nPrediction: NO, Patient is unlikely to have CKD")
    print(f"Confidence : {probability[0][0]*100:.2f}%")

Age:  70
BP:  90
Albumin (al):  0
Sugar (su):  0
Blood Glucose Random (bgr):  144
Blood Urea (bu):  125
Serum Creatinine (sc):  4
Sodium (sod):  136
Potassium (pot):  4.6
Hemoglobin (hrmo):  12
Packed Cell Volume (pcv):  37
White Blood Cell Count (wc):  8200
Red Blood Cell Count (rc):  4.5
sg_b (0/1):  0
sg_c (0/1):  0
sg_d (0/1):  1
sg_e (0/1):  0
rbc_normal (0/1):  1
pc_normal (0/1):  1
pcc_present (0/1):  0
ba_present (0/1):  0
htn_yes (0/1):  1
dm_yes (0/1):  1
cad_yes (0/1):  0
appet_yes (0/1):  0
pe_yes (0/1):  1
ane_yes (0/1):  0



Prediction: YES, Patient is likely to have CKD
Confidence : 79.13%
